# Phase 1 — Embedding extraction (internal MSK-CHORD cohort)

This notebook is the GPU step of Phase 1. It turns the one-prompt-per-patient
table into embeddings for the two extractors evaluated on MSK-CHORD:

* **MedCPT** (`ncbi/MedCPT-Query-Encoder`) — a BERT-style encoder used as a
  de-risking / comparison probe.
* **MedGemma-1.5-4B** (`google/medgemma-1.5-4b-it`) — the primary extractor, the
  only one that beats the tabular baseline.

Running this locally is slow (CPU is the bottleneck), so it is meant for Colab or
any GPU environment. Each section writes a `<model>__ctx_v1__mean__by_patient.parquet`
(one vector per patient) that is scored locally on CPU with
`scripts/22_cox_grid.py`. The protocol (masked-mean pooling, `max_length=512`,
template `ctx_v1`) is identical across models for an apples-to-apples comparison.

## Prerequisites

1. Build the prompts locally and upload `prompts.parquet` to this session:
   ```bash
   python scripts/02_make_prompts.py   # -> data/interim/prompts.parquet
   ```
2. For MedGemma only: accept the model license and create a read token (links in
   the MedGemma section below).

Set `Runtime -> Change runtime type -> GPU (T4)` before running.

In [ ]:
# Colab: install the dependencies for the forward pass.
!pip install -q -U transformers accelerate huggingface_hub pandas pyarrow

In [ ]:
# Optional: mount Google Drive so inputs/outputs persist across runtime restarts.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import numpy as np
import pandas as pd

# Extraction protocol — keep these IDENTICAL across every extractor and cohort
# so all embedding sets are apples-to-apples (same as the local CPU runs).
MAX_LENGTH = 512    # token cap; matches config/models.yaml defaults
BATCH_SIZE = 32     # lower for large models (e.g. 8 for MedGemma on a T4)
POOLING    = "mean" # masked mean over real tokens
TEMPLATE   = "ctx_v1"  # prompt template id; part of the output filename

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def masked_mean(last_hidden, attention_mask):
    """Mean over real (non-pad) tokens -> one vector per sequence."""
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


def save_by_patient(mat, patient_ids, fname):
    """Write the {PATIENT_ID, e0..e{d-1}} parquet that scripts/22_cox_grid.py and
    scripts/41_external_validate.py expect."""
    out = pd.DataFrame(mat, columns=[f"e{i}" for i in range(mat.shape[1])])
    out.insert(0, "PATIENT_ID", np.asarray(patient_ids))
    out.to_parquet(fname, index=False)
    print("wrote", fname, "->", out.shape)
    return out

# Input: the prompts parquet from scripts/02_make_prompts.py.
PROMPTS = "/content/drive/MyDrive/prompts.parquet"

prompts = pd.read_parquet(PROMPTS)
assert {"PATIENT_ID", "prompt", "template_id"} <= set(prompts.columns)
texts = prompts["prompt"].tolist()
ids = prompts["PATIENT_ID"].to_numpy()
print(len(texts), "prompts | template:", prompts["template_id"].iloc[0])

## 1. MedCPT (de-risking probe)

In [ ]:
from transformers import AutoModel, AutoTokenizer

# MedCPT-Query-Encoder is a BERT-style encoder: AutoModel + last_hidden_state,
# float32 (same as the local extractor in src/embedbiomarker/embeddings.py).
CPT_ID = "ncbi/MedCPT-Query-Encoder"
cpt_tok = AutoTokenizer.from_pretrained(CPT_ID)
cpt_model = AutoModel.from_pretrained(CPT_ID).to(DEVICE).eval()


@torch.no_grad()
def embed_medcpt(texts):
    vecs = []
    for i in range(0, len(texts), BATCH_SIZE):
        enc = cpt_tok(texts[i:i + BATCH_SIZE], return_tensors="pt",
                      padding=True, truncation=True, max_length=MAX_LENGTH).to(DEVICE)
        h = cpt_model(**enc).last_hidden_state
        vecs.append(masked_mean(h, enc["attention_mask"]).float().cpu())
    return torch.cat(vecs).numpy()


mat = embed_medcpt(texts)
save_by_patient(mat, ids, f"medcpt__{TEMPLATE}__{POOLING}__by_patient.parquet")

## 2. MedGemma-1.5-4B (primary extractor)

In [ ]:
from huggingface_hub import login
from transformers import AutoModelForImageTextToText, AutoTokenizer

# MedGemma-1.5-4B is the primary extractor (the only one that beats the tabular
# baseline). Accept the license at https://huggingface.co/google/medgemma-1.5-4b-it
# and create a read token at https://huggingface.co/settings/tokens, then paste it.
login()

GEMMA_ID  = "google/medgemma-1.5-4b-it"
GEMMA_TAG = "medgemma15"   # matches config/models.yaml and the local pipeline
GEMMA_BATCH = 8            # smaller batch; the 4B model is heavier than MedCPT

gemma_tok = AutoTokenizer.from_pretrained(GEMMA_ID)
gemma_model = AutoModelForImageTextToText.from_pretrained(
    GEMMA_ID, torch_dtype=torch.bfloat16, device_map="cuda",
).eval()


def get_text_backbone(m):
    """Locate the text transformer inside the multimodal wrapper, so we pool the
    TEXT last_hidden_state (not the multimodal head)."""
    for path in ("model.language_model", "language_model", "model.model.language_model"):
        obj = m
        try:
            for p in path.split("."):
                obj = getattr(obj, p)
            return obj
        except AttributeError:
            continue
    raise AttributeError("text backbone not found; fall back to output_hidden_states=True")


gemma_text = get_text_backbone(gemma_model)


@torch.no_grad()
def embed_medgemma(texts):
    vecs = []
    for i in range(0, len(texts), GEMMA_BATCH):
        enc = gemma_tok(texts[i:i + GEMMA_BATCH], return_tensors="pt",
                        padding=True, truncation=True, max_length=MAX_LENGTH).to("cuda")
        h = gemma_text(input_ids=enc["input_ids"],
                       attention_mask=enc["attention_mask"]).last_hidden_state
        vecs.append(masked_mean(h, enc["attention_mask"]).float().cpu())
    return torch.cat(vecs).numpy()


mat = embed_medgemma(texts)
save_by_patient(mat, ids, f"{GEMMA_TAG}__{TEMPLATE}__{POOLING}__by_patient.parquet")

## Next step

Download the `*__by_patient.parquet` files into `data/processed/embeddings/` on
your machine, then score them (CPU, ~1 min each):

```bash
python scripts/22_cox_grid.py --model medcpt     --pooling mean
python scripts/22_cox_grid.py --model medgemma15 --pooling mean --n-boot 1000 \
    --out "$PWD/results/embedding_grid__medgemma15.json"
```

In [ ]:
# Direct browser download (alternative to saving into Drive).
from google.colab import files
files.download(f"{GEMMA_TAG}__{TEMPLATE}__{POOLING}__by_patient.parquet")